# A Simple Forward and Adjoint Detector Response

This tutorial computes the response of a detector region to a point source directly and through an adjoint inner product. It distills the forward/adjoint workflow in a one-dimensional slab calculation.

## Compute the forward response

A unit point source drives a one-group slab. The forward detector response is the scalar flux integrated over the detector logical volume.

In [1]:
from pathlib import Path

from mpi4py import MPI
from pyopensn.aquad import GLProductQuadrature1DSlab
from pyopensn.context import Finalize
from pyopensn.fieldfunc import FieldFunctionInterpolationVolume
from pyopensn.logvol import RPPLogicalVolume
from pyopensn.mesh import OrthogonalMeshGenerator
from pyopensn.response import ResponseEvaluator
from pyopensn.solver import DiscreteOrdinatesProblem, SteadyStateSourceSolver
from pyopensn.source import PointSource, VolumetricSource
from pyopensn.xs import MultiGroupXS

comm = MPI.COMM_WORLD
rank = comm.rank
nodes = [10.0 * i / 100.0 for i in range(101)]
mesh = OrthogonalMeshGenerator(node_sets=[nodes]).Execute()
mesh.SetUniformBlockID(0)
detector_region = RPPLogicalVolume(infx=True, infy=True, zmin=9.0, zmax=9.8)

xs = MultiGroupXS()
xs.CreateSimpleOneGroup(sigma_t=1.0, c=0.5)
point_source = PointSource(location=[0.0, 0.0, 1.0], strength=[1.0])
quadrature = GLProductQuadrature1DSlab(n_polar=64, scattering_order=0)
problem = DiscreteOrdinatesProblem(
    mesh=mesh,
    num_groups=1,
    groupsets=[
        {
            "groups_from_to": (0, 0),
            "angular_quadrature": quadrature,
            "inner_linear_method": "petsc_gmres",
            "l_abs_tol": 1.0e-10,
        }
    ],
    xs_map=[{"block_ids": [0], "xs": xs}],
    point_sources=[point_source],
)
solver = SteadyStateSourceSolver(problem=problem)
solver.Initialize()
solver.Execute()

scalar_flux = problem.GetScalarFluxFieldFunction(only_scalar_flux=True)[0]
detector = FieldFunctionInterpolationVolume()
detector.SetOperationType("sum")
detector.SetLogicalVolume(detector_region)
detector.AddFieldFunction(scalar_flux)
detector.Execute()
forward_response = float(detector.GetValue())

OpenSn version 1.0.1
2026-08-24 17:09:47 Running OpenSn with 1 processes.

[0]  Done checking cell-center-to-face orientations
[0]  00:00:00.0 Establishing cell connectivity.
[0]  00:00:00.0 Vertex cell subscriptions complete.
[0]  00:00:00.0 Surpassing cell 10 of 100 (10%)
[0]  00:00:00.0 Surpassing cell 20 of 100 (20%)
[0]  00:00:00.0 Surpassing cell 31 of 100 (30%)
[0]  00:00:00.0 Surpassing cell 40 of 100 (40%)
[0]  00:00:00.0 Surpassing cell 50 of 100 (50%)
[0]  00:00:00.0 Surpassing cell 61 of 100 (60%)
[0]  00:00:00.0 Surpassing cell 71 of 100 (70%)
[0]  00:00:00.0 Surpassing cell 80 of 100 (80%)
[0]  00:00:00.0 Surpassing cell 90 of 100 (90%)
[0]  00:00:00.0 Surpassing cell 100 of 100 (100%)
[0]  00:00:00.0 Establishing cell boundary connectivity.
[0]  00:00:00.0 Done establishing cell connectivity.
[0]  Number of cells per partition (max,min,avg) = 100,100,100
[0]  
[0]  Mesh statistics:
[0]    Global cell count             : 100
[0]    Local cell count (avg,max,min): 100,100,

## Compute the adjoint response

The detector weighting becomes the adjoint source. After the adjoint solve, `ResponseEvaluator` evaluates the saved adjoint flux moments at the original point source. The two response estimates should agree within the discretization error.

In [2]:
problem.SetAdjoint(True)
adjoint_source = VolumetricSource(logical_volume=detector_region, group_strength=[1.0])
problem.SetVolumetricSources(volumetric_sources=[adjoint_source])
solver.Execute()

flux_prefix = "tutorial_adjoint_flux_p"
problem.WriteFluxMoments(flux_prefix)
evaluator = ResponseEvaluator(problem=problem)
evaluator.SetOptions(
    buffers=[{"name": "detector", "file_prefixes": {"flux_moments": flux_prefix}}],
    sources={"point": [point_source]},
)
adjoint_response = float(evaluator.EvaluateResponse("detector"))
relative_difference = abs(adjoint_response - forward_response) / abs(forward_response)
if rank == 0:
    print(f"Forward detector response={forward_response:.6e}")
    print(f"Adjoint detector response={adjoint_response:.6e}")
    print(f"Forward-adjoint relative difference={relative_difference:.6e}")
assert relative_difference < 5.0e-3

comm.Barrier()
Path(f"{flux_prefix}{rank}.h5").unlink(missing_ok=True)
comm.Barrier()
if "opensn_console" not in globals():
    from IPython import get_ipython
    if get_ipython() is not None:
        Finalize()
        MPI.Finalize()

[0]  Volumetric source #0 has 8 total subscribing cells.
Forward detector response=1.095097e-04
Adjoint detector response=1.095097e-04
Forward-adjoint relative difference=3.589913e-08
[0]  00:00:00.0 Starting solver execution SteadyStateSourceSolver.
[0]  00:00:00.0 WGS groups [0-0] iteration = 0, residual = 1.000000e+00
[0]  00:00:00.0 WGS groups [0-0] iteration = 1, residual = 1.219073e-01
[0]  00:00:00.0 WGS groups [0-0] iteration = 2, residual = 2.027670e-02
[0]  00:00:00.0 WGS groups [0-0] iteration = 3, residual = 3.273325e-03
[0]  00:00:00.0 WGS groups [0-0] iteration = 4, residual = 5.536413e-04
[0]  00:00:00.0 WGS groups [0-0] iteration = 5, residual = 9.967430e-05
[0]  00:00:00.0 WGS groups [0-0] iteration = 6, residual = 1.576360e-05
[0]  00:00:00.0 WGS groups [0-0] iteration = 7, residual = 2.829501e-06
[0]  00:00:00.0 WGS groups [0-0] iteration = 8, residual = 4.120518e-07
[0]  00:00:00.0 WGS groups [0-0] iteration = 9, residual = 5.289264e-08
[0]  00:00:00.0 WGS groups [0